# Day 09. Exercise 02
# Metrics

## 0. Imports

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
import joblib

## 1. Preprocessing

1. Create the same dataframe as in the previous exercise.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [2]:
df_raw = pd.read_csv('../data/day-of-week-not-scaled.csv')
df_target = pd.read_csv('../data/dayofweek.csv')
df_raw['dayofweek'] = df_target['dayofweek']

In [3]:
X = df_raw.drop(columns=['dayofweek'])
y = df_raw['dayofweek']

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=21, stratify=y
)

## 2. SVM

1. Use the best parameters from the previous exercise and train the model of SVM.
2. You need to calculate `accuracy`, `precision`, `recall`, `ROC AUC`.

 - `precision` and `recall` should be calculated for each class (use `average='weighted'`)
 - `ROC AUC` should be calculated for each class against any other class (all possible pairwise combinations) and then weighted average should be applied for the final metric
 - the code in the cell should display the result as below:

```
accuracy is 0.88757
precision is 0.89267
recall is 0.88757
roc_auc is 0.97878
```

In [5]:
svm_best_params = {
    'C': 10,
    'class_weight': None,
    'gamma': 'auto',
    'kernel': 'rbf',
    'probability': True,
    'random_state': 21
}

In [6]:
svm_model = SVC(**svm_best_params)
svm_model.fit(X_train, y_train)
y_pred_svm = svm_model.predict(X_test)
y_proba_svm = svm_model.predict_proba(X_test)

In [7]:
acc_svm = accuracy_score(y_test, y_pred_svm)
prec_svm = precision_score(y_test, y_pred_svm, average='weighted')
rec_svm = recall_score(y_test, y_pred_svm, average='weighted')
roc_svm = roc_auc_score(y_test, y_proba_svm, multi_class='ovo', average='weighted')

In [8]:
print("SVM metrics:")
print(f"accuracy is {acc_svm:.5f}")
print(f"precision is {prec_svm:.5f}")
print(f"recall is {rec_svm:.5f}")
print(f"roc_auc is {roc_svm:.5f}\n")

SVM metrics:
accuracy is 0.88757
precision is 0.89267
recall is 0.88757
roc_auc is 0.97878



## 3. Decision tree

1. The same task for decision tree

In [9]:
tree_best_params = {
    'class_weight': 'balanced',
    'criterion': 'gini',
    'max_depth': 21,
    'random_state': 21
}

In [10]:
tree_model = DecisionTreeClassifier(**tree_best_params)
tree_model.fit(X_train, y_train)
y_pred_tree = tree_model.predict(X_test)
y_proba_tree = tree_model.predict_proba(X_test)

In [11]:
acc_tree = accuracy_score(y_test, y_pred_tree)
prec_tree = precision_score(y_test, y_pred_tree, average='weighted')
rec_tree = recall_score(y_test, y_pred_tree, average='weighted')
roc_tree = roc_auc_score(y_test, y_proba_tree, multi_class='ovo', average='weighted')

In [12]:
print("Decision Tree metrics:")
print(f"accuracy is {acc_tree:.5f}")
print(f"precision is {prec_tree:.5f}")
print(f"recall is {rec_tree:.5f}")
print(f"roc_auc is {roc_tree:.5f}\n")

Decision Tree metrics:
accuracy is 0.88462
precision is 0.88765
recall is 0.88462
roc_auc is 0.93528



## 4. Random forest

1. The same task for random forest.

In [13]:
rf_best_params = {
    'class_weight': 'balanced',
    'criterion': 'entropy',
    'max_depth': 24,
    'n_estimators': 100,
    'random_state': 21
}

In [14]:
rf_model = RandomForestClassifier(**rf_best_params)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
y_proba_rf = rf_model.predict_proba(X_test)

In [15]:
acc_rf = accuracy_score(y_test, y_pred_rf)
prec_rf = precision_score(y_test, y_pred_rf, average='weighted')
rec_rf = recall_score(y_test, y_pred_rf, average='weighted')
roc_rf = roc_auc_score(y_test, y_proba_rf, multi_class='ovo', average='weighted')

In [16]:
print("Random Forest metrics:")
print(f"accuracy is {acc_rf:.5f}")
print(f"precision is {prec_rf:.5f}")
print(f"recall is {rec_rf:.5f}")
print(f"roc_auc is {roc_rf:.5f}\n")

Random Forest metrics:
accuracy is 0.92899
precision is 0.93051
recall is 0.92899
roc_auc is 0.98934



## 5. Predictions

1. Choose the best model.
2. Analyze: for which `weekday` your model makes the most errors (in % of the total number of samples of that class in your full dataset), for which `labname` and for which `users`.
3. Save the model.

In [17]:
best_model = rf_model
print(f"Best model: Random Forest (accuracy {acc_rf:.5f})")

Best model: Random Forest (accuracy 0.92899)


In [18]:
errors = (y_pred_rf != y_test)
error_df = X_test[errors].copy()
error_df['true'] = y_test[errors]
error_df['pred'] = y_pred_rf[errors]

In [19]:
error_weekday = error_df.groupby('true').size()
total_weekday = y_test.value_counts()
error_pct_weekday = (error_weekday / total_weekday * 100).fillna(0)
print("\nError percentage per weekday (0=Monday,...,6=Sunday):")
print(error_pct_weekday.sort_values(ascending=False))


Error percentage per weekday (0=Monday,...,6=Sunday):
0    22.222222
4    14.285714
5     9.259259
2     6.666667
1     5.454545
3     3.750000
6     2.816901
dtype: float64


In [20]:
labname_cols = [col for col in X.columns if col.startswith('labname_')]
error_labname = {}
total_labname = {}
for col in labname_cols:
    total_labname[col] = (X_test[col] == 1).sum()
    error_labname[col] = (error_df[col] == 1).sum()

In [21]:
error_pct_labname = pd.Series({
    col.replace('labname_', ''): (error_labname[col] / total_labname[col] * 100) if total_labname[col] > 0 else 0
    for col in labname_cols
}).sort_values(ascending=False)
print("\nError percentage per labname:")
print(error_pct_labname)


Error percentage per labname:
lab03       100.000000
laba04       17.142857
lab05s       16.666667
laba06       11.111111
laba04s       8.000000
code_rvw      7.692308
laba06s       6.666667
project1      5.376344
laba05        2.127660
lab02         0.000000
lab03s        0.000000
dtype: float64


In [22]:
user_cols = [col for col in X.columns if col.startswith('uid_user_')]
error_user = {}
total_user = {}
for col in user_cols:
    total_user[col] = (X_test[col] == 1).sum()
    error_user[col] = (error_df[col] == 1).sum()

In [23]:
error_pct_user = pd.Series({
    col.replace('uid_', ''): (error_user[col] / total_user[col] * 100) if total_user[col] > 0 else 0
    for col in user_cols
}).sort_values(ascending=False)
print("\nError percentage per user (top 10):")
print(error_pct_user.head(10))


Error percentage per user (top 10):
user_6     50.000000
user_16    20.000000
user_27    16.666667
user_18    16.666667
user_3     14.285714
user_30    12.500000
user_31    11.111111
user_2     10.714286
user_19    10.526316
user_24     9.090909
dtype: float64


In [24]:
joblib.dump(best_model, '../data/best_model_metrics.joblib')
print("\nModel saved to '../data/best_model_metrics.joblib'")


Model saved to '../data/best_model_metrics.joblib'


## 6. Function

1. Write a function that takes a list of different models and a corresponding list of parameters (dicts) and returns a dict that contains all the 4 metrics for each model.

In [25]:
def compute_metrics(models_params, X_train, y_train, X_test, y_test):
    """
    models_params: list of tuples (model_class, params_dict, name)
    Returns a dict with model names as keys and dict of metrics as values.
    """
    results = {}
    for model_class, params, name in models_params:
        model = model_class(**params)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)
        
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average='weighted')
        rec = recall_score(y_test, y_pred, average='weighted')
        roc = roc_auc_score(y_test, y_proba, multi_class='ovo', average='weighted')
        
        results[name] = {
            'accuracy': acc,
            'precision': prec,
            'recall': rec,
            'roc_auc': roc
        }
    return results

In [26]:
models_list = [
     (SVC, svm_best_params, 'SVM'),
     (DecisionTreeClassifier, tree_best_params, 'DecisionTree'),
     (RandomForestClassifier, rf_best_params, 'RandomForest')
]
metrics_dict = compute_metrics(models_list, X_train, y_train, X_test, y_test)
print(metrics_dict)

{'SVM': {'accuracy': 0.8875739644970414, 'precision': 0.8926729169690374, 'recall': 0.8875739644970414, 'roc_auc': 0.9787793228216216}, 'DecisionTree': {'accuracy': 0.8846153846153846, 'precision': 0.8876518218623483, 'recall': 0.8846153846153846, 'roc_auc': 0.935280206669359}, 'RandomForest': {'accuracy': 0.9289940828402367, 'precision': 0.9305077389964602, 'recall': 0.9289940828402367, 'roc_auc': 0.9893370337145164}}
